In [16]:
import pandas as pd
import snowflake.connector
from dotenv import load_dotenv
import os

load_dotenv()

con = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    warehouse="COMPUTE_WH",
    database="RIDESHARE",
    schema="MARTS"
)
print("Connected to Snowflake")

Connected to Snowflake


In [17]:
def query(sql):
    cursor = con.cursor()
    cursor.execute(sql)
    cols = [col[0] for col in cursor.description]
    rows = cursor.fetchall()
    return pd.DataFrame(rows, columns=cols)

In [18]:
import plotly.express as px

## Uber vs Lyft - Volume and Revenue Comparison

Which company dominates NYC rideshare, and by how much?

In [19]:
df = query("""
    select
        company,
        count(*) as total_trips,
        round(sum(total_fare), 2) as gross_revenue,
        round(sum(total_fare - driver_pay), 2) as platform_revenue,
        round(avg(total_fare), 2) as avg_fare_per_trip
    from fct_trips
    group by company
""")

df['GROSS_REVENUE'] = df['GROSS_REVENUE'].map('${:,.2f}'.format)
df['PLATFORM_REVENUE'] = df['PLATFORM_REVENUE'].map('${:,.2f}'.format)
df

,COMPANY,TOTAL_TRIPS,GROSS_REVENUE,PLATFORM_REVENUE,AVG_FARE_PER_TRIP
0,Uber,13564036,"$356,808,525.53","$120,105,305.74",26.31
1,Lyft,4898054,"$118,548,815.74","$45,348,420.75",24.20


Analysis shows Uber completely dominating NYC rideshare, accounting for 73% of all rides with riders paying an average of $26.31 per Uber trip compared to $24.20 on Lyft. Although, when it comes to revenue percentage, Lyft keeps a 38% margin while Uber sits at 33.7%. Regardless, Uber is certainly controlling the majority of rides in New York. Brand recognition certainly plays a factor in that success. Uber launched two years before Lyft, expanded internationally far earlier, and evolved beyond rides entirely with Uber Eats — meaning customers don't need to request a ride to stay engaged with the app.

## Demand Patterns

### 1.
What time of the day is busiest?

In [21]:
df = query("""
    select
        to_char(time_from_parts(pickup_hour, 0, 0), 'HH12 AM') as hour_of_day,
        count(*) as trip_volume
    from fct_trips
    group by 1
    order by 2 desc
""")

df['TRIP_VOLUME'] = df['TRIP_VOLUME'].map('{:,}'.format)
df

,HOUR_OF_DAY,TRIP_VOLUME
0,06 PM,"1,155,425"
1,07 PM,"1,122,720"
2,05 PM,"1,085,888"
3,08 PM,"1,019,165"
4,09 PM,"973,566"
5,10 PM,"958,042"
6,04 PM,"950,019"
7,03 PM,"922,560"
8,08 AM,"908,565"
9,02 PM,"896,767"


Data shows the late afternoon as the busiest time for requested rides, with 6pm having the most volume. Notably alluding to riders getting off work and possibly rather paying for a ride than walking home. Which also could explain why early mornings contain the lowest volume of requests, with 4am having the least rides. Either less requests or less drivers working then.

### 2.
Weekday demand compared to weekend demand, by borough

In [26]:
df = query("""
    select
        pickup_borough,
        day_type,
        count(*) as trip_volume,
            round(count(*) * 100.0 / sum(count(*)) over (
                                    partition by day_type
        ), 2) as pct_of_day_week
    from fct_trips
    where pickup_borough not in ('N/A', 'EWR')
    group by 1, 2
    order by 1, 2, 3 desc
""")

df['TRIP_VOLUME'] = df['TRIP_VOLUME'].map('{:,}'.format)
df['PCT_OF_DAY_WEEK'] = df['PCT_OF_DAY_WEEK'].map('{}%'.format)
df

,PICKUP_BOROUGH,DAY_TYPE,TRIP_VOLUME,PCT_OF_DAY_WEEK
0,Bronx,Weekday,"1,889,454",11.81%
1,Bronx,Weekend,"305,986",12.40%
2,Brooklyn,Weekday,"4,148,936",25.94%
3,Brooklyn,Weekend,"631,420",25.59%
4,Manhattan,Weekday,"6,620,147",41.39%
5,Manhattan,Weekend,"952,216",38.59%
6,Queens,Weekday,"3,121,016",19.51%
7,Queens,Weekend,"542,395",21.98%
8,Staten Island,Weekday,"214,296",1.34%
9,Staten Island,Weekend,"35,234",1.43%
